# Phase 4: PaDiM baseline, configuration C1 on Sheet Metal

**CRISP-DM stage:** Modelling

The first run from start to finish: learn from good parts, set the pass-or-fail line, inspect half A, and report. Settings come from `configs/C1_padim_resnet18.yaml`.

**Runs on:** Kaggle with the GPU and internet on, to install anomalib and fetch the pretrained network. On the laptop it only checks the code, with an untrained network, a tiny image size and a few sample photos, so its numbers mean nothing.

**Half B is never read.** Half A's part numbers are written here because Kaggle cannot read files from the laptop, and the code stops if the photo count is wrong.

**Why not anomalib's `Engine`:** it runs the fit by itself, but PaDiM's statistics must run on the processor, because the GPU's many-at-once inverse crashes in this setup. Its default run also rescales scores and applies its own threshold, and this project needs the benchmark's rule on raw scores. A short loop using anomalib's model and metrics keeps every step visible. Using `Engine` with a changed fit step would hide more.

**Figures with dataset images:** files named `gallery_*` show real parts, so they stay on the laptop and out of git. Each figure is saved and closed, not displayed.

In [ ]:
!pip install anomalib==2.7.0

## 1. Settings

Each value matches `configs/C1_padim_resnet18.yaml`, which gives the reason for each one. On the laptop, the code compares them with that file.

In [ ]:
import csv
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from anomalib.data import MVTecAD2
from anomalib.metrics.aupro import _AUPRO
from anomalib.models import Padim
from anomalib.models.image.padim.torch_model import PadimModel
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Subset

USING_GPU = torch.cuda.is_available()

PRODUCT = "sheet_metal"
BACKBONE = "resnet18"
LAYERS = ["layer1", "layer2", "layer3"]
SEED = 42
BATCH_SIZE = 8
# Half A, from half_a_files.csv. Every lighting version of each part is used
HALF_A_PARTS = {"good": ["001", "002"], "bad": ["000", "004", "005", "007", "009", "012", "013"]}
HEAT_MAP_PARTS = [("bad", "000"), ("bad", "004"), ("bad", "009"), ("bad", "012"), ("good", "001")]
FALSE_POSITIVE_LIMITS = [0.05, 0.30]
# The benchmark's rule: mean plus 3 standard deviations
STD_MULTIPLIER = 3
FIGURE_DPI = 150

if USING_GPU:
    DEVICE = "cuda"
    INPUT_FOLDER = Path("/kaggle/input")
    IMAGE_SIZE = 256
    PRETRAINED = True
    EXPECTED_HALF_A_PHOTOS = 54
else:
    # The laptop only proves the code runs
    DEVICE = "cpu"
    INPUT_FOLDER = Path("data/sample")
    IMAGE_SIZE = 64
    PRETRAINED = False
    EXPECTED_HALF_A_PHOTOS = 2

config_file = Path("configs/C1_padim_resnet18.yaml")
if config_file.exists():
    import yaml
    with open(config_file, encoding="utf-8") as file:
        config = yaml.safe_load(file)
    if config["method"]["backbone"] != BACKBONE or config["method"]["layers"] != LAYERS:
        raise ValueError("Settings differ from the configuration file")
    if config["seeds"] != [SEED] or config["method"]["train_batch_size"] != BATCH_SIZE:
        raise ValueError("Settings differ from the configuration file")
    print("Settings match the configuration file")

torch.manual_seed(SEED)

product_folder = None
for train_folder in INPUT_FOLDER.rglob("train"):
    if train_folder.parent.name == PRODUCT:
        product_folder = train_folder.parent
if product_folder is None:
    raise FileNotFoundError("Could not find " + PRODUCT)

# root is the outer product folder and category the inner one. prepare_data is not called, as it could try to download
datamodule = MVTecAD2(root=product_folder.parent, category=PRODUCT, num_workers=0)
datamodule.setup()

# The default recipe: squash to a square, then ImageNet normalization
recipe = Padim.configure_pre_processor(image_size=(IMAGE_SIZE, IMAGE_SIZE)).transform

half_a_positions = []
test_paths = datamodule.test_public_data.samples["image_path"].tolist()
for position in range(len(test_paths)):
    path = Path(test_paths[position])
    if path.stem.split("_")[0] in HALF_A_PARTS[path.parent.name]:
        half_a_positions.append(position)
if len(half_a_positions) != EXPECTED_HALF_A_PHOTOS:
    raise ValueError("Wrong number of half A photos: " + str(len(half_a_positions)))

print("Training photos:", len(datamodule.train_data))
print("Validation photos:", len(datamodule.val_data))
print("Half A photos:", len(half_a_positions))

## 2. Helper functions, checked on known answers

- `benchmark_threshold`: the paper's rule. It takes the anomaly value of every pixel in the good validation photos, and returns their mean plus 3 standard deviations. The paper does not say clearly whether pixels are pooled or taken per location, and pooling is the simpler reading.
- `f1_from_counts`: F1 balances catching defects against false alarms. With `tp` correct alarms, `fp` false alarms and `fn` missed, it is `2 x tp / (2 x tp + fp + fn)`.
- `make_loader`: builds batches using the dataset's own `collate_fn`.
- `peak_gpu_gb`: the most GPU memory used since the last reset.
- `roc_auc_score` from scikit-learn compares scores by their order only, so their size does not matter. torchmetrics' `BinaryAUROC` was not used, because it squashes scores outside 0 to 1 with a sigmoid. PaDiM's scores run far above 1, so every photo became a tie and the answer was always 0.5. The known-answer test uses scores above 1 to guard against this.

In [ ]:
def benchmark_threshold(values):
    """Return the mean plus 3 standard deviations of the given values."""
    return float(values.mean() + STD_MULTIPLIER * values.std())


def f1_from_counts(true_positives, false_positives, false_negatives):
    """Return F1, or 0 if nothing was flagged and nothing was there to find."""
    bottom = 2 * true_positives + false_positives + false_negatives
    if bottom == 0:
        return 0.0
    return 2 * true_positives / bottom


def f1_score(predicted, actual):
    """Return F1 from true or false predictions and answers."""
    true_positives = int((predicted & actual).sum())
    false_positives = int((predicted & ~actual).sum())
    false_negatives = int((~predicted & actual).sum())
    return f1_from_counts(true_positives, false_positives, false_negatives)


def make_loader(dataset, positions=None):
    """Build batches in a fixed order, optionally from chosen photos only."""
    source = dataset
    if positions is not None:
        source = Subset(dataset, positions)
    return DataLoader(source, batch_size=BATCH_SIZE, collate_fn=dataset.collate_fn, shuffle=False)


def wait_for_gpu():
    """Wait for the GPU to finish, so the timer does not stop early."""
    if USING_GPU:
        torch.cuda.synchronize()


def peak_gpu_gb():
    """Return the most GPU memory used since the last reset."""
    if USING_GPU:
        return round(torch.cuda.max_memory_allocated() / 1e9, 2)
    return "no gpu"


def reset_peak():
    """Start a new peak memory measurement."""
    if USING_GPU:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


# Known answers: mean 2 plus 3 x standard deviation 1 gives 5
if benchmark_threshold(torch.tensor([1.0, 2.0, 3.0])) != 5.0:
    raise ValueError("Threshold sum is wrong")
# 1 correct alarm, 1 false alarm and 1 missed gives 2 / (2 + 1 + 1) = 0.5
test_predicted = torch.tensor([True, True, False, False])
test_actual = torch.tensor([True, False, True, False])
if f1_score(test_predicted, test_actual) != 0.5:
    raise ValueError("F1 is wrong")
# Scores above 1, in the right order. Known answer: every defective photo ranks higher, so 1.0
if roc_auc_score([0, 0, 1, 1], [20.0, 30.0, 40.0, 50.0]) != 1.0:
    raise ValueError("AU-ROC is wrong")
print("Known-answer tests passed")

## 3. Learn from the good training photos

- `PadimModel` holds the network. In training mode, each batch's features are stored in `memory_bank`.
- `backbone`, `layers` and `pre_trained` are explained in the configuration file. `n_features` is left out, so anomalib's default of 100 for ResNet-18 applies.
- `feature_extractor.eval()` keeps the pretrained network unchanged.
- **The processor route:** features move to the processor, and anomalib's own `gaussian.fit` works out the average and variation per position, then the inverse. The answers then move back to the GPU. This was checked on the real model at size 256.

In [ ]:
model = PadimModel(backbone=BACKBONE, layers=LAYERS, pre_trained=PRETRAINED).to(DEVICE)
model.train()
model.feature_extractor.eval()

reset_peak()
started = time.perf_counter()
for batch in make_loader(datamodule.train_data):
    model(recipe(batch.image).to(DEVICE))
wait_for_gpu()
seconds_features = time.perf_counter() - started
peak_features = peak_gpu_gb()

embeddings = torch.vstack(model.memory_bank).cpu()
model.memory_bank = []

started = time.perf_counter()
model.gaussian.fit(embeddings)
seconds_fit = time.perf_counter() - started
del embeddings

# The answers are needed on the GPU for scoring
model.gaussian.to(DEVICE)
model.eval()
print("Features, seconds:", round(seconds_features, 2))
print("Fit on the processor, seconds:", round(seconds_fit, 2))

## 4. Score photos, and set the benchmark threshold

`score_photos` returns each photo's anomaly map, its score and its mask. PaDiM's photo score is the highest value in its map. anomalib returns scores as a column, shaped `[photos, 1]`, so `squeeze(1)` turns them into a simple list. Without this, comparing scores with labels would make a table of every score against every answer, and image F1 would be wrong with no error.

**Timing:** only the model's work is timed, per photo, because reading files depends on the disk. One extra pass comes first, because the first pass includes set-up work.

**The threshold comes from good validation photos only.** The paper uses one threshold for both levels: it sets it from per-pixel values, and a photo is rejected if at least one pixel exceeds it.

In [ ]:
def score_photos(loader):
    """Run the model on photos. Return maps, scores, masks, paths and model seconds."""
    maps = []
    scores = []
    masks = []
    paths = []
    model_seconds = 0.0
    with torch.no_grad():
        for batch in loader:
            images, batch_masks = recipe(batch.image, batch.gt_mask)
            images = images.to(DEVICE)
            wait_for_gpu()
            started = time.perf_counter()
            output = model(images)
            wait_for_gpu()
            model_seconds = model_seconds + (time.perf_counter() - started)
            # The map has an extra dimension for its one channel
            maps.append(output.anomaly_map.squeeze(1).cpu())
            # Turns the column of scores into a simple list, to line up with the labels
            scores.append(output.pred_score.squeeze(1).cpu())
            masks.append(batch_masks)
            paths.extend(batch.image_path)
    return torch.cat(maps), torch.cat(scores), torch.cat(masks), paths, model_seconds


# The first pass includes set-up work, so it does not count for timing
with torch.no_grad():
    model(torch.zeros(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE))

validation_maps, validation_scores, ignored_masks, ignored_paths, ignored_seconds = score_photos(make_loader(datamodule.val_data))
threshold = benchmark_threshold(validation_maps.flatten())
validation_rejected = int((validation_scores > threshold).sum())
print("Benchmark threshold:", round(threshold, 4))
print("Validation photos above it:", validation_rejected, "of", len(validation_scores))

## 5. Inspect half A and work out the metrics

- **AU-PRO** scores each defect region, up to a limit on false alarms among good pixels. `_AUPRO` is the calculation inside anomalib's metric, used directly because this notebook has plain numbers and not anomalib's batch objects. It was used the same way in the earlier made-up test.
- **Pixel F1 and image F1** use the benchmark threshold.
- **Image AU-ROC** asks whether defective photos usually score higher than good ones. It needs no threshold. The project's rules include it, but the paper does not report it.
- **Flag-everything baseline:** calling every photo defective catches every defect, so image F1 must beat this to mean anything.
- **`regular` lighting alone** matches training, so it shows image-level results without a lighting change.

These are half A numbers, for development. They are not final results, and one run with one seed compares nothing. Masks are at the model's size of 256, and the paper does not say clearly at which size it scores.

In [ ]:
reset_peak()
half_a_loader = make_loader(datamodule.test_public_data, half_a_positions)
half_a_maps, half_a_scores, half_a_masks, half_a_paths, half_a_seconds = score_photos(half_a_loader)
peak_scoring = peak_gpu_gb()

is_defective = []
is_regular = []
for path in half_a_paths:
    is_defective.append(Path(path).parent.name == "bad")
    is_regular.append(Path(path).stem.endswith("_regular"))
is_defective = torch.tensor(is_defective)
is_regular = torch.tensor(is_regular)

# A check on the shapes: a mismatch would pair scores with labels wrongly, with no error
if half_a_scores.shape != is_defective.shape:
    raise ValueError("Scores and labels differ in shape: " + str(list(half_a_scores.shape)) + ", " + str(list(is_defective.shape)))

# A check on the data: good photos must have empty masks, and defective ones must not
for position in range(len(half_a_paths)):
    has_defect_pixels = bool(half_a_masks[position].any())
    if has_defect_pixels != bool(is_defective[position]):
        raise ValueError("Mask does not fit its label: " + half_a_paths[position])


def image_metrics(scores, labels):
    """Return image F1, AU-ROC and the flag-everything F1 for chosen photos."""
    f1 = f1_score(scores > threshold, labels)
    everything = f1_score(torch.ones_like(labels), labels)
    # AU-ROC needs both good and defective photos
    if labels.all() or not labels.any():
        return f1, "not defined", everything
    return f1, float(roc_auc_score(labels.int().tolist(), scores.tolist())), everything


result_lines = []
result_lines.append("configuration: C1, PaDiM, " + BACKBONE + ", " + PRODUCT + ", seed " + str(SEED))
result_lines.append("image size: " + str(IMAGE_SIZE) + ", squashed. Half A only, every lighting condition")
result_lines.append("photos: training " + str(len(datamodule.train_data)) + ", validation " + str(len(datamodule.val_data))
                    + ", half A " + str(len(half_a_paths)) + " (" + str(int(is_defective.sum())) + " defective)")
result_lines.append("pretrained network: " + str(PRETRAINED))
result_lines.append("benchmark threshold, mean plus 3 standard deviations of good validation pixels: " + str(round(threshold, 4)))
result_lines.append("good validation photos above the threshold: " + str(validation_rejected) + " of " + str(len(validation_scores)))

for limit in FALSE_POSITIVE_LIMITS:
    aupro = _AUPRO(fpr_limit=limit)
    aupro.update(half_a_maps, half_a_masks.long())
    result_lines.append("AU-PRO at " + str(round(limit * 100)) + " percent: " + str(round(float(aupro.compute()), 4)))

pixel_f1 = f1_score(half_a_maps > threshold, half_a_masks)
result_lines.append("pixel F1: " + str(round(pixel_f1, 4)))

groups = [("all lighting", torch.ones_like(is_regular)), ("regular lighting only", is_regular)]
for group_name, chosen in groups:
    f1, auroc_value, everything = image_metrics(half_a_scores[chosen], is_defective[chosen])
    if auroc_value != "not defined":
        auroc_value = round(auroc_value, 4)
    result_lines.append(group_name + ": photos " + str(int(chosen.sum())) + ", image F1 " + str(round(f1, 4))
                        + ", flag-everything F1 " + str(round(everything, 4)) + ", image AU-ROC " + str(auroc_value))

predicted_defective = half_a_scores > threshold
# A check on the counts: flagged photos can never exceed the photos in that group
if int((predicted_defective & is_defective).sum()) > int(is_defective.sum()):
    raise ValueError("More defective photos flagged than exist")
if int((predicted_defective & ~is_defective).sum()) > int((~is_defective).sum()):
    raise ValueError("More good photos flagged than exist")
result_lines.append("all lighting, defective photos flagged: " + str(int((predicted_defective & is_defective).sum())) + " of " + str(int(is_defective.sum())))
result_lines.append("all lighting, good photos flagged: " + str(int((predicted_defective & ~is_defective).sum())) + " of " + str(int((~is_defective).sum())))

result_lines.append("seconds per photo, model only: " + str(round(half_a_seconds / len(half_a_paths), 4)))
result_lines.append("seconds for features, training photos: " + str(round(seconds_features, 2)))
result_lines.append("seconds for the fit, on the processor: " + str(round(seconds_fit, 2)))
result_lines.append("peak GPU memory GB, features: " + str(peak_features))
result_lines.append("peak GPU memory GB, scoring: " + str(peak_scoring))

# Numbers only, with no image content, so these files can be committed
with open("c1_sheet_metal_metrics.txt", "w") as results_file:
    for line in result_lines:
        results_file.write(line + "\n")

with open("c1_sheet_metal_photo_scores.csv", "w", newline="") as scores_file:
    writer = csv.writer(scores_file)
    writer.writerow(["file_name", "label", "score", "above_threshold"])
    for position in range(len(half_a_paths)):
        path = Path(half_a_paths[position])
        writer.writerow([path.name, path.parent.name, round(float(half_a_scores[position]), 4), bool(predicted_defective[position])])

for line in result_lines:
    print(line)

## 6. Heat maps

The photos are fixed in advance, by part number and in `regular` lighting, so the figure cannot flatter the result.

- **Left:** the photo with normalization undone, with its mask outlined in red.
- **Right:** PaDiM's heat map, darker where it is more suspicious. Every map shares one scale, so they can be compared. The black outline marks pixels above the benchmark threshold, and the red outline is the mask.

In [ ]:
normalize_step = recipe.transforms[1]
means = torch.tensor(normalize_step.mean).view(3, 1, 1)
stds = torch.tensor(normalize_step.std).view(3, 1, 1)

chosen_panels = []
for position in range(len(half_a_paths)):
    path = Path(half_a_paths[position])
    for label, part in HEAT_MAP_PARTS:
        if path.parent.name == label and path.name == part + "_regular.png":
            chosen_panels.append(position)

# One shared scale, so every map is read the same way
shared_top = float(half_a_maps.max())
shared_bottom = float(half_a_maps.min())

if len(chosen_panels) > 0:
    figure, axes = plt.subplots(len(chosen_panels), 2, figsize=(9, 4.2 * len(chosen_panels)), squeeze=False)
    for row_number in range(len(chosen_panels)):
        position = chosen_panels[row_number]
        path = Path(half_a_paths[position])
        mask = half_a_masks[position].float()
        heat_map = half_a_maps[position]

        dataset_position = half_a_positions[position]
        photo = recipe(datamodule.test_public_data[dataset_position].image)
        photo = (photo * stds + means).clamp(0, 1)

        axes[row_number][0].imshow(photo[0], cmap="gray", vmin=0, vmax=1)
        axes[row_number][1].imshow(heat_map, cmap="Blues", vmin=shared_bottom, vmax=shared_top)
        if bool(mask.any()):
            axes[row_number][0].contour(mask, levels=[0.5], colors="red", linewidths=0.8)
            axes[row_number][1].contour(mask, levels=[0.5], colors="red", linewidths=0.8)
        if float(heat_map.max()) > threshold:
            axes[row_number][1].contour(heat_map, levels=[threshold], colors="black", linewidths=0.8)
        axes[row_number][0].set_title(path.parent.name + " " + path.name, fontsize=10)
        axes[row_number][1].set_title("Score " + str(round(float(half_a_scores[position]), 2))
                                      + ", threshold " + str(round(threshold, 2)), fontsize=10)
        axes[row_number][0].axis("off")
        axes[row_number][1].axis("off")
    figure.suptitle("MVTec AD 2, " + PRODUCT + " (CC BY-NC-SA 4.0), half A, C1. Red: mask. Black: above threshold", y=1.0)
    figure.tight_layout()
    figure.savefig("gallery_heatmaps_C1_" + PRODUCT + ".png", dpi=FIGURE_DPI, bbox_inches="tight")
    # Closing without showing keeps dataset images out of the notebook's log
    plt.close(figure)
print("Heat map photos:", len(chosen_panels))
print("Finished")